## Imports

In [ ]:
import numpy as np
import xarray as xr
import copy
import src.utils
import src.lme_utils
import matplotlib as mpl
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
import cmocean
import os
import pathlib
import cartopy.crs as ccrs
import pandas as pd
import scipy.stats
import tqdm
import time
import xesmf as xe

## Set seaborn plotting style
sns.set(rc={"axes.facecolor": "white", "axes.grid": False})
sns.set_palette("colorblind")

## initialize RNG
rng = np.random.default_rng(seed=100)

## Funcs

In [ ]:
## specify lons/lats for E/W boxes
LONS_W = [120, 200]
LONS_E = [200, 280]


## funcs to plot boxes
def plot_wbox(ax, **kwargs):
    src.utils.plot_box(ax, lons=LONS_W, lats=[-5, 5], **kwargs)


def plot_ebox(ax, **kwargs):
    src.utils.plot_box(ax, lons=LONS_E, lats=[-5, 5], **kwargs)


## funcs to average over regions
def get_eq_avg(data, lons):
    """average over equatorial region and longitudes"""
    return data.sel(longitude=slice(*lons), latitude=slice(-5, 5)).mean(
        ["longitude", "latitude"]
    )


def get_e(data):
    return get_eq_avg(data, LONS_E)


def get_w(data):
    return get_eq_avg(data, LONS_W)


def get_dx(data):
    return get_e(data) - get_w(data)


def add_gridlines(axs):
    """func to add gridlines to axs object"""

    for ax in axs[:-1, 0]:
        ax.gridlines(
            crs=ccrs.PlateCarree(),
            draw_labels=True,
            linewidth=1,
            alpha=0,
            xlocs=[],
            ylocs=[-20, 0, 20],
        )
    for ax in axs[-1]:
        gl_ = ax.gridlines(
            crs=ccrs.PlateCarree(),
            draw_labels=True,
            linewidth=1,
            alpha=0,
            xlocs=[50, 120 - 160, -80],
            # xlocs=[],
            ylocs=[-20, 0, 20],
        )
        gl_.top_labels = False
        # gl_.bottom_labels = True
    # gl_.left_labels = False

    return


def plot_level(ax, data, level, ls="-", c="white"):
    """plot single level on hovmoller"""
    ax.contour(
        data.longitude,
        data.latitude,
        data,
        levels=[level],
        colors=c,
        linestyles=ls,
        zorder=10,
        transform=ccrs.PlateCarree(),
    )
    return


import cartopy.crs as ccrs


def make_cb_range(dx, nlevs):
    amp = dx * nlevs - dx / 2

    lev1 = np.arange(0, amp + dx / 2, dx) + dx / 2
    lev0 = -lev1[::-1]
    return np.concatenate([lev0, lev1])


def plot_sst2(ax, data, dx=0.2, nlev=5):
    """plot data on ax"""

    cp = ax.contourf(
        data.longitude,
        data.latitude,
        data,
        cmap="cmo.balance",
        levels=make_cb_range(dx, nlev),
        transform=ccrs.PlateCarree(),
        extend="both",
    )

    return cp


def plot_sst_sigma(ax, data, lev_min=0.3, dx=0.2, nlev=5):
    """plot data on ax"""

    cp = ax.contourf(
        data.longitude,
        data.latitude,
        data,
        cmap="cmo.amp",
        levels=np.arange(lev_min, lev_min + dx * (nlev + 1), dx),
        transform=ccrs.PlateCarree(),
        extend="both",
    )

    return cp

## Data loading

In [ ]:
def sel_area(data, lon_range=None, lat_range=None):
    """select area on t grids"""

    ## find in lon range
    if lon_range is None:
        lon_idx = np.array([True for _ in data.nlon])
    else:
        lon_idx = (
            ((data.TLONG >= lon_range[0]) & (data.TLONG <= lon_range[1]))
            .any("nlat")
            .values
        )

    if lat_range is None:
        lat_idx = np.array([True for _ in data.nlat])
    else:
        lat_idx = (
            ((data.TLAT >= lat_range[0]) & (data.TLAT <= lat_range[1]))
            .any("nlon")
            .values
        )

    return data.isel(nlon=lon_idx, nlat=lat_idx)


def area_avg(data, varname, lon_range=None, lat_range=None):
    """average over area"""

    ## trim data
    data_ = sel_area(data, lon_range=lon_range, lat_range=lat_range)

    ## get dims to sum over
    integrate = lambda x: (x * data_["dA"]).sum(["nlon", "nlat"])

    return integrate(data_[varname]) / integrate(1.0)


def get_eli_helper(exceeds_thresh):
    """compute ELI from mask of threshold exceedance"""

    ## sum and count longitudes exceeding thresh
    lon = exceeds_thresh.longitude
    longitude_sum = (exceeds_thresh * lon).sum(["longitude", "latitude"])
    longitude_count = exceeds_thresh.sum(["longitude", "latitude"])

    ## eli is average longitude
    eli = longitude_sum / longitude_count

    return eli


def get_eli(rsst, rsst_thresh=0, max_lat=15):
    """compute ELI from tropical SST data"""

    ## get equatorial Pac. SST
    rsst_pac = rsst.sel(longitude=slice(120, 280), latitude=slice(-max_lat, max_lat))

    ## get boolean array where SST exceeds thresh
    exceeds_thresh0 = rsst_pac >= rsst_thresh

    ## compute initial ELI estimate
    eli0 = get_eli_helper(exceeds_thresh0)

    return eli0

## Load the data

### init. cluster

In [ ]:
from dask.distributed import LocalCluster, Client

cluster = LocalCluster(n_workers=2)
client = Client(cluster)
client

### do loading

Get time-mean SST

In [ ]:
# SAVE_FP = pathlib.Path("/glade/work/kcarr/lme_data")
# SAVE_FNAME = SAVE_FP / "sst_regridded_forced.nc"

# if SAVE_FNAME.is_file():
#     forced = xr.open_dataset(SAVE_FNAME).compute()

# else:

#     d = xr.open_dataset(SAVE_FP / "sst_regridded.nc").compute()
#     d = d.rename({"lat": "latitude", "lon": "longitude", "SST": "sst"})

#     ## compute tropical SST
#     d["Ttrop"] = d["SST"].mean(["latitude", "longitude"])

#     ## compute relative SST
#     d["rsst"] = d["sst"] - d["Ttrop"]

#     ## ensemble mean
#     forced = d.mean("member")

#     ## save
#     forced.to_netcdf(SAVE_FP / "sst_regridded_forced.nc")

# ## get time-mean
# mu = forced["SST"].mean("time")

Get forced component of the rest and add to time-mean to get total forced

In [ ]:
SAVE_FP_SNR = pathlib.Path("/glade/work/kcarr/lme_data/snr")

## load
time_coder = xr.coders.CFDatetimeCoder(use_cftime=True)
sst = xr.open_dataset(SAVE_FP_SNR / "sst_split.nc", decode_times=time_coder)
sst = sst.rename({"lat": "latitude", "lon": "longitude"})

## get forced components
forced = sst["forced"]

## compute RSST
forced_rsst = forced - forced.mean(["latitude", "longitude"])
forced = xr.merge([forced.rename("sst"), forced_rsst.rename("rsst")])

## Analysis

In [ ]:
## get climatology
clim = forced.isel(time=slice(None, 360)).mean("time")

## compute ELI
forced["eli"] = get_eli(forced["rsst"], max_lat=20, rsst_thresh=0)

## compute other indices
forced["Tw"] = get_w(forced["rsst"])
forced["Te"] = get_e(forced["rsst"])
forced["dTdx"] = forced["Te"] - forced["Tw"]

Compute annual mean

In [ ]:
forced_ann = forced.resample({"time": "YS"}).mean()
forced_ann_clim = forced_ann.sel(time=slice("0900-01", "1000-01")).mean("time")

### Spatial plot

In [ ]:
mpl.rcParams["figure.dpi"] = 100

In [ ]:
## specify intervals
dxs = np.array([0.8, 0.4])

fig = plt.figure(figsize=(6, 2.25), layout="constrained")
format_func = lambda ax,: src.utils.plot_setup_pac(ax, max_lat=21, lon_range=(30, 285))
axs = src.utils.subplots_with_proj(fig, nrows=1, ncols=1, format_func=format_func)

cp00 = plot_sst2(axs[0, 0], clim["rsst"], dx=dxs[0])
# cp10 = plot_sst2(axs[1, 0], enso_pattern0, dx=dxs[1])
# cp01 = plot_sst_sigma(axs[1, 0], sst_sigma0, dx=dxs[1])

## add formatting to axs
for ax in axs.flatten():
    plot_wbox(ax, c="gray", lw=1.5)
    plot_ebox(ax, c="gray", lw=1.5)

## label
axs[0, 0].set_title("(a) Relative SST mean", loc="left")
# axs[1, 0].set_title(
#     r"(b) SST ENSO pattern", loc="left"
# )

## label regions
for loc, label, c in zip([(160, 7), (240, 7)], [r"$T_{wp}$", r"$T_{ct}$"], ["w", "k"]):
    axs[0, 0].text(x=loc[0], y=loc[1], s=label, color=c, transform=ccrs.PlateCarree())

## add labels
add_gridlines(axs)
bbox = dict(boxstyle="round", facecolor="white", alpha=1)
legend_kwargs = dict(x=0.01, y=0.02, fontdict=dict(size=12, color="gray"), bbox=bbox)
for ax, dx in zip(axs.flatten(), dxs.flatten()):
    # ax.text(s=f"Interval: {dx} K", transform=ax.transAxes, **legend_kwargs)
    ax.set_aspect("auto")

## colorbars
cb_kwargs = dict(label=r"$^{\circ}$C", pad=0.03)
fig.colorbar(cp00, ax=axs[0, 0], ticks=[-3.6, 0, 3.6], **cb_kwargs)
# fig.colorbar(cp10, ax=axs[1,0], ticks=[-1.8,0,1.8], **cb_kwargs)

plt.show()

#### ELI

In [ ]:
volc_forcing = src.lme_utils.load_volc_forcing()

In [ ]:
fig, ax = plt.subplots(figsize=(12, 3))

## plot data
ax.plot(forced_ann.time.dt.year, forced_ann["eli"])
ax.axhline(forced_ann_clim["eli"], ls="--", c="k", lw=1)
ax.set_xticks([850, 1258, 2005])

ax.scatter(
    volc_forcing.year,
    forced_ann_clim["eli"] * xr.ones_like(volc_forcing["Global_norm"]),
    s=200 * volc_forcing["Global_norm"],
    c="magenta",
    edgecolor="k",
    marker="^",
    zorder=10,
)

ax.set_ylabel(r"Warm pool extent ($^{\circ}$E)")
ax.set_xlim([840, 2015])

plt.show()

### Hovmoller (longitude)

In [ ]:
def format_ax(ax):
    """format single ax"""

    ## format func
    ax.set_xlim([120, 285])
    ax_kwargs = dict(ls="--", c="w", lw=0.8)
    ax.axvline(200, **ax_kwargs)
    # ax.axvline(230, **ax_kwargs)
    ax.axhline(1850, **ax_kwargs)
    ax.set_xticks([])
    ax.set_yticks([])

    return ax


def format_axs(axs):
    """format all axs"""

    ## format/labeling
    for ax in axs.flatten():
        format_ax(ax)

    ## titles
    # axs[0, 0].set_title("El Niño")
    # axs[0, 1].set_title("La Niña")

    ## xlabels
    for ax in axs:
        ax.set_xticks([120, 200, 280])
        ax.set_xlabel(r"Longitude")

    ## ylabels
    for ax in axs:
        ax.set_yticks([865, 1850, 1991])
        # ax.set_ylabel("Year")

    return

#### Compute

In [ ]:
eq_avg = lambda x: x.sel(latitude=slice(-20, 20)).mean("latitude")
forced_eq = eq_avg(forced).resample({"time": "YS"}).mean()

#### Plot

In [ ]:
fig, ax = plt.subplots(figsize=(3.5, 5), layout="constrained")

plot_kwargs = dict(cmap="cmo.balance", extend="both")

## plot total
cp = ax.contourf(
    forced_eq.longitude,
    forced_eq.time.dt.year,
    forced_eq["rsst"]
    - forced_eq["rsst"].sel(time=slice("0900-01", "0999-12")).mean("time"),
    levels=src.utils.make_cb_range(0.5, 0.05),
    **plot_kwargs,
)

ax.set_xlim([120, 280])
format_ax(ax)
ax.set_yticks([])
ax.set_ylabel("Year")
cb = fig.colorbar(cp, ticks=[-3, 0, 3], label="RSST (K)", fraction=0.05)

## plot ELI
eli_kwargs = dict(c="k", lw=1.5)
ax.plot(
    forced_eq["eli"],
    forced_eq.time.dt.year,
    **eli_kwargs,
)

ax.scatter(
    (-10 + forced_ann_clim["eli"]) * xr.ones_like(volc_forcing["Global_norm"]),
    volc_forcing.year,
    s=200 * volc_forcing["Global_norm"],
    c="magenta",
    edgecolor="k",
    marker=">",
    zorder=10,
)

ax.set_yticks([850, 1258, 2005])

plt.show()

### Hovmoller (latitude)

In [ ]:
# wp_avg = lambda x: x.sel(longitude=slice(120, 200)).mean("longitude")
wp_avg = lambda x: x.sel(longitude=slice(200, 280)).mean("longitude")
forced_wp = wp_avg(forced).resample({"time": "YS"}).mean()
forced_wp_anom = forced_wp - forced_wp.sel(time=slice("0900-01", "0999-12")).mean(
    "time"
)

In [ ]:
fig, axs = plt.subplots(2, 1, figsize=(7, 7), layout="constrained")

plot_kwargs = dict(cmap="cmo.balance", extend="both")

## plot
cps = []
for ax, f, amp in zip(axs, [forced_wp, forced_wp_anom], [3, 0.2]):

    ## plot total
    cps.append(
        ax.contourf(
            f.time.dt.year,
            f.latitude,
            f["rsst"].transpose("latitude", "time"),
            levels=src.utils.make_cb_range(amp, amp / 10),
            **plot_kwargs,
        )
    )

    ax.set_ylabel("Latitude")
    ax.set_yticks([-20, -10, 0, 10, 20])
    ax.set_ylim([-20, 20])
    cb = fig.colorbar(
        cps[-1], ax=ax, ticks=[-amp, 0, amp], label="RSST (K)", fraction=0.05
    )
    ax.axhline(0, ls="--", c="k", lw=1)

    ## volcanoes
    ax.scatter(
        volc_forcing.year,
        xr.zeros_like(volc_forcing["Global_norm"]),
        s=200 * volc_forcing["Global_norm"],
        c="magenta",
        edgecolor="k",
        marker="^",
        zorder=10,
    )

axs[0].set_xticks([])
axs[0].set_title("Total")
axs[1].set_title("Diff")

plt.show()

### Scatter plots

In [ ]:
sel = lambda x: x.resample({"time": "YS"}).mean()
# sel = lambda x : x.resample({"time":"QS-DEC"}).mean().isel(time=slice(2,None,4))

fig, ax = plt.subplots(figsize=(3, 3), layout="constrained")

ax.scatter(
    sel(forced["eli"]),
    sel(forced["Te"]),
    s=3,
)
ax.axhline(0, ls="--", c="k", lw=1)
ax.axvline(190, ls="--", c="k", lw=1)
ax.set_xlabel("ELI")
ax.set_ylabel(r"$T_{ct}$")
ax.set_xticks([190, 200])

plt.show()

In [ ]:
# sel = lambda x : x.resample({"time":"YS"}).mean()
sel = lambda x: x.resample({"time": "QS-DEC"}).mean().isel(time=slice(2, None, 4))

fig, axs = plt.subplots(1, 3, figsize=(9, 3), layout="constrained")
for ax, v in zip(axs, ["Tw", "Te", "dTdx"]):
    print(scipy.stats.pearsonr(sel(forced["eli"]).values, sel(forced[v]).values)[0])
    ax.scatter(
        sel(forced["eli"]),
        sel(forced[v]),
        s=5,
    )

# src.utils.set_lims(axs)
axs[-1].set_ylim(axs[-1].get_ylim()[::-1])

plt.show()